In [ ]:
# IMPORT LIBRARIES

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
# INITIALIZE JS FOR FORCE PLOTS
# Useful for Jupyter Notebook / JupyterLab
shap.initjs()

# Binary Classification

In [ ]:
data = load_breast_cancer()

In [ ]:
X = pd.DataFrame(data.data,columns=data.feature_names)

y = pd.Series(data.target,name="target")

In [ ]:
print("X:", X.shape)
print("y:", y.shape)

In [ ]:
print(data.target_names)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

In [ ]:
binary_model = RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1)

In [ ]:
binary_model.fit(X_train,y_train)

In [ ]:
binary_predictions = binary_model.predict(X_test)

In [ ]:
print("\nBinary Model Accuracy:")
print(accuracy_score(y_test, binary_predictions))

In [ ]:
print("\nClassification Report:")
print(classification_report(y_test,binary_predictions,target_names=data.target_names))

In [ ]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test,binary_predictions))

## CREATE SHAP EXPLAINER

In [ ]:
binary_explainer = shap.TreeExplainer(binary_model)

## CALCULATE SHAP VALUES

In [ ]:
binary_shap_values = binary_explainer(X_test)

print("\nBinary SHAP Explanation:")
print(binary_shap_values)

print("\nSHAP value shape:")
print(binary_shap_values.values.shape)

### IMPORTANT:
    For modern SHAP versions, TreeExplainer(model)(X)
    returns a shap.Explanation.
    
    For a binary RandomForest model, the output is commonly:
    
         samples × features
    
    in the current Explanation-based API.
    
    Older SHAP versions may return:
    
         list[class_0_values, class_1_values]
    
    Therefore, the code below handles both styles.

### SELECT BINARY CLASS TO EXPLAIN

 We will explain CLASS 1.

 In the breast cancer dataset:

 class 0 = malignant
 class 1 = benign

In [ ]:
selected_binary_class = 1

print("\nSelected binary class:")
print(selected_binary_class,"->",data.target_names[selected_binary_class])

### HELPER FUNCTION FOR BINARY SHAP VALUES

In [ ]:
def get_binary_class_shap_values(explainer,X_data,shap_explanation,class_id=1):
    """
    Return a shap.Explanation for the selected binary class.

    Handles common SHAP output formats.
    """

    values = shap_explanation.values

    # --------------------------------------------------------
    # Case 1:
    # Modern Explanation:
    #
    # samples × features
    # --------------------------------------------------------

    if values.ndim == 2:

        return shap.Explanation(
            values=values,
            base_values=shap_explanation.base_values,
            data=shap_explanation.data,
            feature_names=shap_explanation.feature_names
        )

    # --------------------------------------------------------
    # Case 2:
    # Some models / versions:
    #
    # samples × features × classes
    # --------------------------------------------------------

    elif values.ndim == 3:

        class_values = values[:, :, class_id]

        base_values = shap_explanation.base_values

        if np.ndim(base_values) == 2:
            class_base_values = base_values[:, class_id]
        else:
            class_base_values = base_values

        return shap.Explanation(
            values=class_values,
            base_values=class_base_values,
            data=X_data.values,
            feature_names=X_data.columns.tolist())

    else:
        raise ValueError("Unexpected binary SHAP shape: "+ str(values.shape))


binary_class_shap = get_binary_class_shap_values(
    binary_explainer,
    X_test,
    binary_shap_values,
    selected_binary_class
)

# GLOBAL EXPLANATION

## 1. BAR PLOT

In [ ]:
shap.plots.bar(
    binary_class_shap,
    max_display=15,
    show=False)

plt.title("SHAP Global Feature Importance - Bar Plot")

plt.tight_layout()
plt.show()

### INTERPRETATION:

 The bar plot uses average absolute SHAP values.

 Large bar:
     Feature has large average impact.

 Small bar:
     Feature has smaller average impact.

 IMPORTANT:

 Bar plot does NOT show whether a feature increases
 or decreases predictions.

## BEESWARM PLOT

In [ ]:
shap.plots.beeswarm(
    binary_class_shap,
    max_display=15,
    show=False)

plt.title("SHAP Global Explanation - Beeswarm")

plt.tight_layout()
plt.show()

### INTERPRETATION:

 X-axis:
     SHAP value

 Right:
     Positive contribution

 Left:
     Negative contribution

 Color:
     Feature value

 Red:
     High feature value

 Blue:
     Low feature value

 Each dot:
     One observation

## HEATMAP
 Use first 100 observations to keep visualization readable.
 
 You can remove [:100] for the complete test set.

In [ ]:
heatmap_values = binary_class_shap[:100]

shap.plots.heatmap(
    heatmap_values,
    max_display=15,
    show=False)

plt.title("SHAP Heatmap")

plt.tight_layout()
plt.show()

### INTERPRETATION:

 Rows:
     Observations

 Columns:
     Features

 Cell:
     SHAP value

 Positive and negative contributions can be seen
 across many observations simultaneously.

# LOCAL EXPLANATION

In [ ]:
# SELECT ONE OBSERVATION

sample_index = 0

print("Explaining test observation:",sample_index)

print("\nActual class:")
print(y_test.iloc[sample_index],"->",data.target_names[y_test.iloc[sample_index]])

print("\nPredicted class:")
print(binary_model.predict(X_test.iloc[[sample_index]])[0],"->",
    data.target_names[
        binary_model.predict(
            X_test.iloc[[sample_index]])[0]])

print("\nPredicted probabilities:")
probabilities = binary_model.predict_proba(X_test.iloc[[sample_index]])

print(probabilities)

## WATERFALL

In [ ]:
shap.plots.waterfall(
    binary_class_shap[sample_index],
    max_display=15,
    show=False)

plt.title("SHAP Waterfall - Individual Prediction")

plt.tight_layout()
plt.show()

### INTERPRETATION:

 Starts from:

     BASE VALUE

 Then each feature pushes the prediction:

     ↑ positive contribution
     ↓ negative contribution

 Finally:

     MODEL OUTPUT

## FORCE PLOT

In [ ]:
force_plot = shap.plots.force(
    binary_class_shap[sample_index]
)

force_plot

In [ ]:
# Alternative older API:

shap.force_plot(binary_explainer.expected_value,
     binary_class_shap.values[sample_index],
     X_test.iloc[sample_index])

## DECISION PLOT

In [ ]:
shap.plots.decision(
    binary_class_shap.base_values[sample_index],
    binary_class_shap.values[sample_index],
    feature_names=X_test.columns.tolist(),
    show=False)

plt.title("SHAP Decision Plot - Individual Prediction")

plt.tight_layout()
plt.show()

# RELATIONSHIP

In [ ]:
# SELECT MOST IMPORTANT FEATURE

mean_abs_shap = np.abs(
    binary_class_shap.values
).mean(axis=0)

most_important_index = np.argmax(
    mean_abs_shap
)

most_important_feature = X_test.columns[most_important_index]

print("Most important feature:",most_important_feature)

## DEPENDENCE / SCATTER PLOT

In [ ]:
shap.plots.scatter(
    binary_class_shap[
        :,
        most_important_feature
    ],
    show=False)

plt.title(
    f"SHAP Dependence / Scatter Plot: "
    f"{most_important_feature}")

plt.tight_layout()
plt.show()


### INTERPRETATION:

 X-axis:
     Actual feature value

 Y-axis:
     SHAP value

 This helps identify:

     Linear relationships
     Nonlinear relationships
     Threshold effects
     Outliers
     Possible interactions

# INTERACTIONS

## SHAP INTERACTION VALUES
TreeExplainer supports interaction values for tree models.

In [ ]:
interaction_values = binary_explainer.shap_interaction_values(X_test)

print("Interaction shape:",np.asarray(interaction_values).shape)

## HANDLE DIFFERENT INTERACTION OUTPUT FORMATS
Older SHAP versions may return:

     list[class_0, class_1]

 while some configurations may return:

     samples × features × features

In [ ]:
interaction_array = interaction_values

if isinstance(interaction_values,list):
    # Select desired class
    interaction_array = interaction_values[elected_binary_class]

else:
    interaction_array = np.asarray(interaction_values)
    # Possible:
    # samples × features × features × classes
    if interaction_array.ndim == 4:
        interaction_array = interaction_array[:,:,:,selected_binary_class]

print("Selected class interaction shape:",interaction_array.shape)

## VISUALIZE INTERACTION FOR ONE OBSERVATION

In [ ]:
sample_interaction = interaction_array[sample_index]

# Find strongest interaction pair.
#
# Remove diagonal because:
#     diagonal = main effect

interaction_matrix = np.abs(sample_interaction.copy())

np.fill_diagonal(interaction_matrix,0)

feature_i, feature_j = np.unravel_index(np.argmax(interaction_matrix),interaction_matrix.shape)

feature_i_name = X_test.columns[feature_i]
feature_j_name = X_test.columns[feature_j]

print("\nStrongest interaction for selected observation:")

print(feature_i_name,"<->",feature_j_name)

print("Interaction SHAP value:",sample_interaction[feature_i,feature_j])

## INTERACTION HEATMAP

In [ ]:
plt.figure(figsize=(12, 10))

plt.imshow(sample_interaction,aspect="auto")

plt.colorbar(label="SHAP interaction value")

plt.xticks(range(len(X_test.columns)),X_test.columns,rotation=90)

plt.yticks(range(len(X_test.columns)),X_test.columns)

plt.title("SHAP Interaction Values - One Observation")

plt.tight_layout()
plt.show()


### INTERPRETATION:

 Diagonal:
     Main feature effect

 Off-diagonal:
     Interaction between two features

 Large positive:
     Features jointly push output upward.

 Large negative:
     Features jointly push output downward.

## GLOBAL INTERACTION IMPORTANCE

Average absolute interaction strength across observations.

In [ ]:
global_interaction = np.mean(np.abs(interaction_array),axis=0)

# Remove main effects from interaction ranking.
np.fill_diagonal(global_interaction,0)

# Find strongest global interaction
global_i, global_j = np.unravel_index(np.argmax(global_interaction),global_interaction.shape)

print("\nStrongest global feature interaction:")

print(X_test.columns[global_i],"<->",X_test.columns[global_j])

print("Mean absolute interaction:",global_interaction[global_i,global_j])